In [1]:
from robo_mvp.etl import AssetSpec, run_etl
from robo_mvp.regime import classify_regime_simple
from robo_mvp.pipeline import run_demo

In [2]:
# 1) Quick Start에서 자산 스펙 정의 (예시)
asset_specs = [
    AssetSpec(asset_id="CASH_BIL", ticker="BIL.US",  asset_name="Cash (BIL)", asset_class="CASH"),
    AssetSpec(asset_id="GLB_EQ_VTI", ticker="VTI.US", asset_name="US Equity",  asset_class="EQUITY"),
    AssetSpec(asset_id="KR_EQ_EWY",  ticker="EWY.US", asset_name="KR Equity",  asset_class="EQUITY"),
    AssetSpec(asset_id="GOLD_GLD",   ticker="GLD.US", asset_name="Gold",       asset_class="COMMOD"),
    AssetSpec(asset_id="LT_BOND_TLT",ticker="TLT.US", asset_name="LT Treasury",asset_class="BOND"),
]

### Quick Start: 자산 스펙 정의
================================
아래 asset_specs는 "우리가 투자 대상으로 사용할 자산 목록"을 정의하는 부분이다.
각 자산은 Stooq에서 데이터를 가져올 수 있어야 하며,
이후 ETL → 국면분류 → 포트폴리오 → XAI 전 과정에서 공통으로 사용된다.

1. asset_id
- 내부적으로 사용하는 고유 ID (임의로 정해도 되지만, 이후 전 과정에서 일관되게 사용됨)
- 추천 규칙:
    자산군_지역(or속성)_대표티커
    예) CASH_BIL, GLB_EQ_VTI, KR_EQ_EWY, GOLD_GLD, LT_BOND_TLT
- 이 값은 feat_asset_monthly, regime, pipeline 전반에서 key 역할을 하므로
  중복되면 안 되고, 문자열로 명확해야 함

2. ticker
- Stooq에서 실제로 데이터를 다운로드할 때 사용하는 티커
- 형식: "티커.US" (대부분의 미국 ETF는 .US)
- 찾는 방법:
    (1) https://stooq.com 접속
    (2) 검색창에 ETF 이름 또는 티커 입력
    (3) 상세 페이지에서 티커 확인 (예: VTI.US, EWY.US, GLD.US, TLT.US, BIL.US)
- ⚠️ 주의:
    Yahoo Finance 티커와 다를 수 있으므로 반드시 Stooq 기준으로 확인

3. asset_name
- 사람이 읽기 쉬운 자산 이름 (리포트, 결과 출력, 설명용)
- 분석/계산에는 직접 사용되지 않지만,
  결과 설명(XAI, 사용자 리포트)에서 매우 중요함
- 예: "US Equity", "KR Equity", "Gold", "LT Treasury"

4. asset_class
- 자산의 대분류 (전략/제약/설명에 사용)
- 권장 값 예시:
    "CASH"    : 현금성 자산 (T-bill, MMF 등)
    "EQUITY"  : 주식
    "BOND"    : 채권
    "COMMOD"  : 원자재 (금 등)
- 이 값은:
    - 국면별 정책 선택
    - 주식 비중 제한(cap)
    - XAI 설명에서 "주식/채권/대체자산 기여도" 해석에 활용됨

[자산 추가 예시]
- 미국 중기채 ETF 추가:
  AssetSpec(asset_id="MID_BOND_IEI", ticker="IEI.US", asset_name="US 3-7Y Treasury", asset_class="BOND")
- 글로벌 주식 ETF 추가:
  AssetSpec(asset_id="GLB_EQ_VT", ticker="VT.US", asset_name="Global Equity", asset_class="EQUITY")

In [3]:
# 2) asset 리스트 확보
asset_master_df, raw_price_daily_df, feat_asset_monthly_df = run_etl(asset_specs)

assets = [a.asset_id for a in asset_specs] 

### ETL: 자산 데이터 수집 및 가공

1. asset_master_df
- 자산 메타데이터 테이블
- 각 자산의 ID, 이름, 자산군, 티커 정보 포함

In [4]:
asset_master_df

,asset_id,asset_name,asset_class,ticker,currency,is_active
0,CASH_BIL,Cash (BIL),CASH,BIL.US,USD,1
1,GLB_EQ_VTI,US Equity,EQUITY,VTI.US,USD,1
2,KR_EQ_EWY,KR Equity,EQUITY,EWY.US,USD,1
3,GOLD_GLD,Gold,COMMOD,GLD.US,USD,1
4,LT_BOND_TLT,LT Treasury,BOND,TLT.US,USD,1


2. raw_price_daily_df
- 자산별 일별 가격 데이터
- 본 mvp에서 활용은 적으나 후에 충분히 사용가능

In [5]:
raw_price_daily_df

,date,open,high,low,close,volume,asset_id
0,2007-05-30,87.1102,87.1302,87.0365,87.1302,1.628510e+03,CASH_BIL
1,2007-05-31,87.1102,87.1302,87.1102,87.1302,1.177283e+04,CASH_BIL
2,2007-06-01,87.1501,87.1501,87.1501,87.1501,1.524157e+03,CASH_BIL
3,2007-06-04,87.1701,87.1701,87.1701,87.1701,1.105741e+03,CASH_BIL
4,2007-06-05,87.1890,87.2079,87.1701,87.1701,3.940332e+03,CASH_BIL
...,...,...,...,...,...,...,...
25668,2025-12-30,87.7400,88.0400,87.6744,87.8600,2.510678e+07,LT_BOND_TLT
25669,2025-12-31,87.6800,87.9100,87.1300,87.1600,3.650004e+07,LT_BOND_TLT
25670,2026-01-02,87.4000,87.4100,87.0201,87.0300,4.073125e+07,LT_BOND_TLT
25671,2026-01-05,87.1800,87.5200,87.1100,87.4600,3.017803e+07,LT_BOND_TLT


3. feat_asset_monthly_df
- 자산별 월 단위 feature 데이터 (핵십 입력 데이터)
- 월말 기준 가격, 수익률, 변동성, 최대낙폭 등 포함
- 국면 분류, 포트폴리오 최적화, XIA 분석의 공통 입력

In [6]:
feat_asset_monthly_df

,eom,px_eom,ret_1m,ret_3m,ret_12m,vol_3m,dd_6m,dd_12m,asset_id
0,2007-05-31,87.1302,NaN,NaN,NaN,NaN,0.000000,0.000000,CASH_BIL
1,2007-06-30,87.5139,0.004404,NaN,NaN,NaN,0.000000,0.000000,CASH_BIL
2,2007-07-31,87.4740,-0.000456,NaN,NaN,NaN,-0.000456,-0.000456,CASH_BIL
3,2007-08-31,87.4939,0.000227,0.004174,NaN,0.002148,-0.000229,-0.000229,CASH_BIL
4,2007-09-30,87.3215,-0.001970,-0.002199,NaN,0.000918,-0.002199,-0.002199,CASH_BIL
...,...,...,...,...,...,...,...,...,...
1228,2025-09-30,89.3700,0.031986,0.012691,-0.059742,0.020045,0.000000,-0.024863,LT_BOND_TLT
1229,2025-10-31,90.2900,0.010294,0.038771,0.004747,0.014674,0.000000,-0.014825,LT_BOND_TLT
1230,2025-11-30,90.2100,-0.000886,0.041686,-0.015698,0.013647,-0.000886,-0.010156,LT_BOND_TLT
1231,2025-12-31,87.1600,-0.033810,-0.024729,0.015843,0.018721,-0.034666,-0.043623,LT_BOND_TLT


In [7]:
# 3) 국면 분류
equity_assets = ["GLB_EQ_VTI", "KR_EQ_EWY"]  # “주식”으로 cap & XAI target에 쓰일 묶음
regime_monthly_df = classify_regime_simple(feat_asset_monthly_df, equity_assets=equity_assets)

### 국면 분류

1. equity_assets
- 주식 자산으로 간주할 리스트
- 시장 국면 판단시, 주식 시장의 상태를 대표하는 자산묶음으로 사용

2. regime_monthly_df
- feat_asset_monthly_df를 받아 주식 자산들의 수익률/변동성/낙폭 정보를 기반으로 월별 시장 국면을 4가지로 분류

In [8]:
regime_monthly_df['regime_code']

0        DEFENSIVE
1        DEFENSIVE
2         RISK_OFF
3          RISK_ON
4        DEFENSIVE
          ...     
247        RISK_ON
248    RISK_ON_VOL
249      DEFENSIVE
250    RISK_ON_VOL
251        RISK_ON
Name: regime_code, Length: 252, dtype: object

In [9]:
# 4) pipeline 실행: profile을 입력으로
report_user, report_opp = run_demo(
    feat_asset_monthly=feat_asset_monthly_df,
    regime_monthly=regime_monthly_df,
    asset_master=asset_master_df,
    demo_eom="2024-12-31", # 포트폴리오 추천할 기준 시점
    user_profile="AGGRESSIVE", # 사용자 성향
    assets=assets,
    background_months=60, # XAI 계산시 사용하는 과거 데이터 길이(개월 수)
    nsamples=200, # SHAP 샘플 수 (설명 정밀도 / 계산 시간 트레이드 오프)
    top_k=3 # 상위 중요 변수 개수
)

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
report_user['meta']

{'eom': '2024-12-31',
 'regime': 'RISK_OFF',
 'profile': 'AGGRESSIVE',
 'policy': 'CAR_RP'}

In [11]:
report_user['portfolio_user']

,asset_id,weight
0,CASH_BIL,0.8352
1,KR_EQ_EWY,0.0474
2,GOLD_GLD,0.0453
3,LT_BOND_TLT,0.0384
4,GLB_EQ_VTI,0.0337


In [12]:
report_user['xai_user']

,entity,metric,value,shap,direction
0,CASH_BIL,vol_3m,0.001721,0.067629,UP
1,GLB_EQ_VTI,vol_3m,0.042601,-0.017195,DOWN
2,LT_BOND_TLT,vol_3m,0.037454,0.001860,UP


In [13]:
report_opp['meta']

{'eom': '2024-12-31',
 'regime': 'RISK_OFF',
 'profile': 'CONSERVATIVE',
 'policy': 'MDD_GUARD'}

In [14]:
report_opp['portfolio_user']

,asset_id,weight
0,LT_BOND_TLT,0.3379
1,GOLD_GLD,0.2746
2,CASH_BIL,0.2375
3,KR_EQ_EWY,0.1224
4,GLB_EQ_VTI,0.0276


In [15]:
report_opp['xai_user']

,entity,metric,value,shap,direction
0,regime,RISK_OFF,1.0,-0.017924,DOWN
1,regime,RISK_ON,0.0,-0.008455,DOWN
2,regime,RISK_ON_VOL,0.0,-0.005820,DOWN


### LLM

제미나이 2.5 사용

In [16]:
from robo_mvp.llm_explainer import explain_reports_with_gemini

narration = explain_reports_with_gemini(
    report_user=report_user,
    report_opp=report_opp,
    asset_master_df=asset_master_df,
    model="gemini-2.5-flash"
)

print(narration)


안녕하세요, 고객님의 자산 관리를 돕는 로보어드바이저 MVP입니다. 이번 달 고객님의 포트폴리오 분석 결과를 바탕으로 쉽고 명확하게 설명해 드리겠습니다.

---

### 1) 한 문장 요약
현재 위험회피 장세로 인해 고객님의 위험감수형 투자 성향에도 불구하고 현금 중심의 매우 방어적인 포트폴리오가 추천되었습니다.

### 2) 이번 달 시장국면 설명
이번 달(2024년 12월 말 기준) 시장 국면은 '위험회피 장세(방어 모드)'로 판단됩니다. 이는 전반적인 시장 불확실성이 높아 투자자들이 위험 자산보다는 안전 자산을 선호하는 경향이 짙음을 의미합니다. 이러한 국면에서는 자산 보존과 변동성 관리에 중점을 둡니다.

### 3) 사용자 성향 포트폴리오 추천 (위험감수형)
**고객님의 위험감수형 포트폴리오**
*   Cash (BIL): 83.52%
*   KR Equity: 4.74%
*   Gold: 4.53%
*   LT Treasury: 3.84%
*   US Equity: 3.37%

**왜 이렇게 배분했는지:**
*   **시장 국면 반영:** 현재 '위험회피 장세'라는 시장 국면이 반영되어, 고객님의 위험감수형 성향에도 불구하고 위험자산 노출을 최소화하고 현금 비중을 대폭 늘렸습니다.
*   **글로벌 주식 변동성 요인:** 글로벌 주식(VT) / 최근 6개월 최대낙폭이 크게 나타나, 주식 비중을 줄이고 현금 비중을 늘리는 데 가장 크게 기여했습니다.
*   **미국 주식 변동성 요인:** 미국 주식(VTI) / 최근 3개월 변동성이 높아지면서, 미국 주식 비중을 줄이는 데 기여하여 전반적인 위험자산 노출이 더욱 제한되었습니다.

### 4) 반대 성향 포트폴리오(비교) (안정형)
**안정형 포트폴리오**
*   LT Treasury: 33.79%
*   Gold: 27.46%
*   Cash (BIL): 23.75%
*   KR Equity: 12.24%
*   US Equity: 2.76%

**무엇이 달라졌는지:**
안정형 투자 성향의 포트폴리오